In [0]:
use catalog kishoredb;
use schema cap_db;

### Task 1 
Incremental Data Loading using copy into command without manual schema



In [0]:
-- creating the table without using schema
create table orders_bronze;

In [0]:
-- Batch streaming data ingestion using the copy into cmd
copy into orders_bronze
from '/Volumes/kishoredb/cap_db/cap_files/incremental_data_files/'
fileformat =json
format_options('header'='true','inferSchema'='true')
copy_options('mergeSchema'='true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
5019,5019,0


In [0]:
-- After adding Jan and feb dataset ingestion using copy into cmd
select count(*) as feb_cnt from orders_bronze ;

feb_cnt
11632


In [0]:
-- After adding mar dataset ingestion using copy into 
select count(*) as mar_cnt from orders_bronze;

mar_cnt
16651


### Task -2
Order Data analysis

In [0]:
-- 1.total no of orders placed 
select count(*) as total_orders from orders_bronze;

total_orders
16651


In [0]:
-- 2.orders by payment method
select payment_method,count(*) as total_orders from orders_bronze group by 1 

payment_method,total_orders
card,4186
UPI,4124
wallet,4140
cash,4201


In [0]:
-- 3.order status distribution
select order_status,count(*) as total_orders from orders_bronze group by order_status

order_status,total_orders
cancelled,4129
en route,4238
delivered,4105
preparing,4179


In [0]:
-- 4.orders per restaurant 
select restaurant_id,count(*) as total_orders from orders_bronze group by restaurant_id 

restaurant_id,total_orders
R060,81
R040,92
R123,100
R098,81
R070,86
R111,76
R136,86
R159,84
R131,108
R089,101


In [0]:
-- 5.orders per customer
select customer_id , count(*) as total_orders from orders_bronze group by customer_id

customer_id,total_orders
C5784,1
C5037,1
C2559,2
C0977,2
C0752,1
C1136,2
C1524,4
C9889,3
C7872,3
C4710,3


In [0]:
-- 6. orders handled per delivery agents
select agent_id,count(*) as total_orders from orders_bronze group by agent_id

agent_id,total_orders
A099,41
A165,34
A352,29
A050,29
A220,40
A360,79
A285,45
A143,28
A124,34
A400,44


In [0]:
-- 7.Top 10 orders by total amount
select order_id,round(sum(total_amount),2) as total_amt from orders_bronze group by 1 order by 2 desc limit 10

order_id,total_amt
c12860c3-683a-47f7-b790-7cbbbcfc0fec,361.46
05e56a7c-190c-4bde-ac13-e34b38fe155a,355.9
341cb220-46e4-4f0d-b2f2-2e494b5729dd,352.45
3b6f5bd2-08e4-412d-be9e-7701463015b0,348.95
f43228bd-94e5-4ef6-914a-c7ebdceaa572,342.91
621d746e-bd7d-4e27-b06e-a098a9bffda7,340.65
75dfeba8-60e0-42c2-9149-315f030e8d9d,333.85
72559328-7d0d-4177-8c44-4ebc43dca731,332.98
b9dc92fc-1922-4274-a42c-85b83e46bea2,327.8
7d9b6852-ffb2-4bac-84e4-cc7960284e30,327.44


In [0]:
-- 8. avg tip per payment method
select payment_method,round(avg(tip),2) as avg_tip  from orders_bronze group by payment_method 

payment_method,avg_tip
card,2.51
UPI,2.53
wallet,2.5
cash,2.47


In [0]:
-- 9.percent of orders  tip >10% of total_amount
select round(avg(case when tip>0.1*total_amount then 1.0 else 0 end) * 100.0 , 2) as percent_orders from orders_bronze;

percent_orders
21.49


In [0]:
-- 10.percentage of success vs cancelled vs failed orders
select round(avg(case when order_status='delivered' then 1.0 else 0 end)*100.0,2) as success_orders_percentage,
       round(sum(case when order_status='cancelled' then 1 else 0 end)*100.0 / count(*),2) as cancelled_orders_percentage,
       round(count(case when order_status='en route' then 1 end)*100.0 / count(*),2) as failed_orders_percentage       
from orders_bronze

success_orders_percentage,cancelled_orders_percentage,failed_orders_percentage
24.65,24.80,25.45


In [0]:
-- 11.top cust (no of orders) and top res (no of orders)
with cte1 as(select customer_id,count(*) as cust_total_orders from orders_bronze group by customer_id order by cust_total_orders desc limit 1),
cte2 as(
select restaurant_id,count(*) as res_total_orders from orders_bronze group by restaurant_id order by 2 desc limit 1)
select * from cte1,cte2;

customer_id,cust_total_orders,restaurant_id,res_total_orders
C7381,9,R005,114


In [0]:
-- 12.Explode items_ordered
select e.item_id,e.item_name,e.price,e.quantity
from orders_bronze
lateral view explode(items_ordered) as e ;

item_id,item_name,price,quantity
I0117,Over Juice,0.0,5
I0329,Job Ice Cream,6.66,3
I0251,Follow Tea,24.58,3
I0052,To Nachos,28.81,3
I0228,Laugh Juice,6.07,1
I0097,Attack Biryani,18.79,5
I0211,Specific Spring Roll,21.36,1
I0255,Finish Garlic Bread,0.0,3
I0382,Magazine Pasta,27.07,5
I0443,He Coffee,27.38,1


In [0]:
-- 13. Most frequntly ordered items
select item,count(*) as no_of_orders from orders_bronze 
lateral view explode(items_ordered.item_name) as item 
group by item order by no_of_orders desc limit 1

item,no_of_orders
Cell Garlic Bread,161


In [0]:
-- 14. Avg no of items per order
select round(avg(item_count),2) as avg_per_order 
from
(select order_id,count(item_id) as item_count from orders_bronze lateral view explode(items_ordered.item_id) as item_id  group by order_id )

avg_per_order
1.65


In [0]:
-- 15.Find item revenue contribution per restaurant
select restaurant_id,e.item_id,round(sum(e.price * e.quantity),2) as item_revenue 
from (select restaurant_id,explode(items_ordered) as e from orders_bronze )
group by restaurant_id,e.item_id order by item_revenue desc

restaurant_id,item_id,item_revenue
R088,I0306,8772.96
R039,I0254,8771.2
R089,I0500,8534.17
R198,I0357,8028.15
R102,I0189,7978.38
R089,I0108,7636.23
R171,I0253,7533.64
R084,I0074,7443.14
R104,I0284,7396.8
R139,I0145,7207.2


### Task 3
order Data and Dimensions dataset analysis

In [0]:

-- 1.Find the preferred restaurant names for each customer -W 
with cte as (select restaurant_id,customer_id ,count(*) as cnt from orders_bronze group by 2,1 order by 2 desc ),
rnk as (select *, row_number() over(partition by customer_id order by cnt desc ) as rn from cte)
select r.customer_id,f.name from rnk r join restaurants_filtered f on f.restaurant_id=r.restaurant_id
where rn=1 

customer_id,name
C0001,Spice Villa Grill
C0003,Spice Villa Grill
C0004,Tasty Bites Corner
C0006,Pasta Palace Grill
C0007,Spice Villa Grill
C0008,Tasty Bites Kitchen
C0009,Curry Leaf Corner
C0011,Curry Leaf Corner
C0013,Pasta Palace Express
C0014,Tasty Bites Express


In [0]:
-- 2.Find the top 10 areas receiving orders in each month
with cte as (select extract(month from o.timestamp) as month,l.area,count(*) as order_cnt from orders_bronze o join locations_filtered l on o.delivery_location_id=l.location_id group by extract(month from o.timestamp), l.area )
,
ranked as (
    select month,area,order_cnt,
     row_number() over(partition by month order by order_cnt desc) as rnk
     from cte
)
select month,area,order_cnt from ranked where rnk<=10
order by month,order_cnt desc

month,area,order_cnt
1,Rodriguez Mill,87
1,Eric Hills,86
1,Hill Station,80
1,Smith Estates,79
1,Harper Street,76
1,Copeland Parks,75
1,Morgan Key,75
1,John Underpass,74
1,Jennifer Wall,73
1,Ashley Extensions,73


In [0]:
-- 3.Find all successfully delivered orders that were paid using a card to analyze payment preferences and delivery success rate
select * from orders_bronze where order_status='delivered' and payment_method='card'

agent_id,customer_id,delivery_location_id,items_ordered,order_id,order_status,payment_method,restaurant_id,timestamp,tip,total_amount
A086,C5605,L089,"List(List(I0491, Talk Pasta, 10.6, 2))",e293e41d-e335-4d6b-8504-34f96140e660,delivered,card,R006,2024-01-15T14:20:24,4.5,21.2
A352,C7525,L082,"List(List(I0403, Outside Juice, 23.61, 1), List(I0018, Finish Nachos, 6.76, 5), List(I0384, Mrs Cake, 9.28, 2))",c8bd7ee7-a451-447a-811b-fd69aff36062,delivered,card,R053,2024-01-04T07:36:42,0.3,75.97
A068,C1909,L083,"List(List(I0038, Near Soda, 15.97, 4), List(I0134, Structure Wrap, 16.48, 3))",e71e1b46-c026-4913-9b58-870fde58e72b,delivered,card,R047,2024-01-13T20:37:01,4.41,113.32
A196,C3246,L022,"List(List(I0185, Today Nachos, 22.46, 5))",ede3312f-d227-42a5-a1d0-d191999abbd1,delivered,card,R136,2024-01-02T20:57:26,1.4,112.3
A202,C8344,L100,"List(List(I0160, Almost Brownie, 13.71, 4), List(I0187, Travel Tea, 7.78, 2))",7e5fb9b0-c7e0-4080-89f3-6707df2a16a4,delivered,card,R074,2024-01-25T00:13:09,3.53,70.4
A390,C3024,L051,"List(List(I0254, Sell Ice Cream, 27.41, 2))",22accf1f-f43d-4051-9240-9a971c27bf85,delivered,card,R039,2024-01-20T00:43:15,1.31,54.82
A072,C2294,L082,"List(List(I0388, Nor Brownie, 9.17, 2))",bee913bc-f4fd-44b7-8f51-229c93ef4c5d,delivered,card,R018,2024-01-19T19:54:39,0.91,18.34
A307,C8811,L024,"List(List(I0112, Run Pizza, 25.36, 1), List(I0049, Soldier Ice Cream, 21.53, 2), List(I0441, Power Biryani, 0.0, 4))",5d838bcb-e367-4734-93fe-c9a2551e440f,delivered,card,R190,2024-01-05T14:06:33,2.46,68.42
A279,C1179,L038,"List(List(I0470, Radio Brownie, 25.75, 2))",938839a3-20ac-4b3b-a9b1-fb9b364bc210,delivered,card,R159,2024-01-02T16:18:53,4.28,51.5
A157,C2504,L062,"List(List(I0066, Reason Cake, 27.3, 1), List(I0158, Third Spring Roll, 18.26, 5))",3cd888ae-1614-4d75-bd5c-037edd3684f4,delivered,card,R110,2024-01-20T11:43:04,4.19,118.6


In [0]:
-- creating a temp table for the cuisines bcoz restaurants having different type of cuisines list
create or replace temporary table restaurants_cuisines_filtered
as 
select restaurant_id,name,explode(cuisine) as cuisines,location_id,rating,delivery_fee 
from (select restaurant_id, name,split(cuisines, ',') AS cuisine,location_id,rating , delivery_fee
from restaurants_filtered);
select * from restaurants_cuisines_filtered ;

restaurant_id,name,cuisines,location_id,rating,delivery_fee
R001,Food Haven Grill,Indian,L069,3.3,3.9
R002,Spice Villa Hub,Thai,L012,3.8,2.25
R002,Spice Villa Hub,Italian,L012,3.8,2.25
R002,Spice Villa Hub,Chinese,L012,3.8,2.25
R004,Curry Leaf Hub,Indian,L048,3.1,3.03
R004,Curry Leaf Hub,American,L048,3.1,3.03
R005,Tasty Bites Grill,Mexican,L039,3.9,1.02
R007,Pasta Palace Corner,American,L014,3.3,3.33
R007,Pasta Palace Corner,Mexican,L014,3.3,3.33
R007,Pasta Palace Corner,Chinese,L014,3.3,3.33


In [0]:
-- 4.Which cuisines generate the highest avg order value 
select r.cuisines,round(avg(o.total_amount),2) as avg_order from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id group by r.cuisines order by avg_order desc limit 1

cuisines,avg_order
Indian,82.28


In [0]:
-- 5.Which delivery agents have delivered the most high-value order (above 50) 
select agent_id, count(distinct order_id) AS high_value_order_count 
from orders_bronze
where total_amount > 50 and order_status='delivered'
group by agent_id
order by high_value_order_count desc;

agent_id,high_value_order_count
A360,20
A056,20
A207,18
A248,18
A230,18
A393,17
A390,15
A245,15
A125,14
A061,14


In [0]:
-- 6.Which cities (area_name) are generating the most revenue for italian restaurant 
select l.area,round(sum(o.total_amount),2) as total_revenue from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id join locations_filtered l on o.delivery_location_id=l.location_id 
where r.cuisines ilike 'Italian' group by l.area order by 2 desc 

area,total_revenue
John Underpass,4077.09
Brown Spring,3811.75
Donald Row,3759.01
Jimmy Center,3659.54
Kirk Valley,3655.7
Padilla Mall,3601.73
Dorothy Forest,3598.22
Eric Hills,3587.16
Karen Cliff,3499.69
Gomez Hollow,3393.3


In [0]:
-- 7. what is the tip-to-total ratio by cusine type
select  r.cuisines,round(sum(o.tip)/sum(o.total_amount),2) as ratio from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id 
group by r.cuisines 

cuisines,ratio
Italian,0.03
Indian,0.03
Thai,0.03
Chinese,0.03
Mexican,0.03
American,0.03


In [0]:
-- 8.Which menu items are most frequently ordered and from which restaurants  
select exploded.item_id, m.item_name, m.restaurant_id, count(*) as times_ordered 
from (
  select o.restaurant_id, explode(o.items_ordered) as exploded
 from orders_bronze o
)
join menu_items_filtered m on exploded.item_id = m.item_id
group by exploded.item_id, m.item_name, m.restaurant_id
order by times_ordered desc

item_id,item_name,restaurant_id,times_ordered
I0357,Argue Cake,R198,111
I0011,Purpose Brownie,R138,104
I0274,Even Tea,R090,103
I0254,Sell Ice Cream,R039,101
I0145,Rich Biryani,R139,101
I0074,Painting Coffee,R084,101
I0255,Finish Garlic Bread,R082,100
I0090,Others Wrap,R165,100
I0087,Student Tea,R158,99
I0189,Market Spring Roll,R102,99


In [0]:
-- 9.which agents have delivered the most diverse cuisine 
select a.agent_id,count(distinct r.cuisines) as unq_cnt from orders_bronze o join delivery_agents_filtered a on o.agent_id=a.agent_id join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id  group by a.agent_id order by unq_cnt desc 

agent_id,unq_cnt
A400,6
A285,6
A395,6
A220,6
A198,6
A352,6
A287,6
A385,6
A124,6
A012,6


In [0]:
-- 10. What is the avg customer lifetime value by signup year  
with cte as (select c.customer_id,extract(year from c.signup_date) as signup_year ,round(sum(o.total_amount),2) as total from orders_bronze o join customers_filtered c on o.customer_id=c.customer_id group by c.customer_id,extract(year from c.signup_date))
select signup_year, round(avg(total),2) as avg_total from cte group by signup_year order by signup_year;

signup_year,avg_total
1954,22.64
1955,172.65
1956,139.9
1957,155.41
1958,135.53
1959,141.08
1960,173.02
1961,222.25
1962,192.06
1963,137.26


In [0]:
-- 11.Which restaurants have the highest order frequency per location 
select r.restaurant_id,l.location_id,count(*) as order_cnt from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id join locations_filtered l on r.location_id=l.location_id group by r.restaurant_id,l.location_id order by order_cnt desc

restaurant_id,location_id,order_cnt
R005,L039,114
R198,L022,111
R171,L093,108
R083,L069,108
R092,L057,106
R185,L095,104
R035,L022,104
R090,L046,103
R146,L047,103
R180,L086,103


In [0]:
-- 12.Which locations are prone to order cancellations
select l.area, count(*) as order_cnt from orders_bronze o join locations_filtered l on o.delivery_location_id=l.location_id where order_status='cancelled' group by l.area order by l.area, order_cnt

area,order_cnt
Adams Fort,37
Aguilar Pass,45
Alex Curve,44
Alex Mission,43
Andrews Heights,45
Angela Lodge,43
Anthony Forge,41
Arnold Fort,50
Ashley Extensions,53
Boyle Dam,35


In [0]:
-- 13.Find top 10 customers who have ordered from the wildest variety of restaurants 
select o.customer_id,count(distinct r.restaurant_id) as variety_res from orders_bronze o join restaurants_filtered r on o.restaurant_id=r.restaurant_id group by o.customer_id order by variety_res desc limit 10

customer_id,variety_res
C7381,7
C5365,7
C6470,7
C7526,7
C1423,6
C0362,6
C6971,6
C2116,6
C4165,6
C1449,6


In [0]:
-- 14. Identify restaurants with frequent repeat customers
select r.restaurant_id,o.customer_id,count(*) as order_cnt from orders_bronze o join restaurants_filtered r on  o.restaurant_id=r.restaurant_id 
group by 1,2 
having order_cnt >1 

restaurant_id,customer_id,order_cnt
R148,C7659,2
R038,C9577,2
R055,C9140,2
R066,C5987,2
R150,C0119,2
R019,C1838,2
R053,C2179,2
R019,C4924,2
R181,C3734,2
R029,C2966,2


In [0]:
-- 15. Which Payment methods are most preferred for high -value orders across cuisines
select r.cuisines,o.payment_method,count(*) as order_cnt from orders_bronze o join restaurants_cuisines_filtered r on o.restaurant_id=r.restaurant_id where o.total_amount >40 
group by r.cuisines,o.payment_method  order by order_cnt desc 

cuisines,payment_method,order_cnt
Mexican,cash,833
American,cash,812
American,wallet,803
American,card,799
American,UPI,794
Mexican,card,775
Mexican,UPI,770
Mexican,wallet,762
Thai,wallet,725
Thai,card,691


In [0]:
drop schema kishoredb.streamdb cascade;
create schema kishoredb.streamdb;